# Manual Media-Source Token Diagnostics

Paste one rebuilt content item as JSON, run the media-source model, and inspect entity and token-level diagnostics.


## Setup

Run this notebook from an environment where `impresso-pipelines[mediasources]` is installed. For local development, the cells below also add the repository root and the sibling `impresso-pipelines` checkout to `sys.path` when present.


In [ ]:
from __future__ import annotations

import hashlib
import html
import importlib
import inspect
import json
import tempfile
import sys
from pathlib import Path
from typing import Any

for stale_name in ("pipe", "result", "entities", "cookbook_row"):
    globals().pop(stale_name, None)

try:
    from IPython.display import HTML, display
except ImportError:
    class HTML(str):
        pass

    def display(value: Any) -> None:
        print(value)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'lib').exists() else NOTEBOOK_DIR.parent
SIBLING_PIPELINES = REPO_ROOT.parent / 'impresso-pipelines'

for path in [REPO_ROOT, SIBLING_PIPELINES]:
    if path.exists():
        path_string = str(path)
        if path_string in sys.path:
            sys.path.remove(path_string)
        sys.path.insert(0, path_string)

import impresso_pipelines
import impresso_pipelines.mediasources.mediasources_pipeline as mediasources_pipeline_module
import lib.cli_mediasources as cli_mediasources_module

mediasources_pipeline_module = importlib.reload(mediasources_pipeline_module)
cli_mediasources_module = importlib.reload(cli_mediasources_module)
MediaSourcesPipeline = mediasources_pipeline_module.MediaSourcesPipeline
MediaSourcesProcessor = cli_mediasources_module.MediaSourcesProcessor

try:
    import torch
    import transformers
except ImportError:
    torch = None
    transformers = None


def text_sha256(value: str) -> str:
    return hashlib.sha256(value.encode('utf-8')).hexdigest()


## Load The Pipeline

Set `LOCAL_FILES_ONLY = False` if the model is not already cached and the notebook environment has Hugging Face network access.


In [ ]:
MODEL_ID = 'impresso-project/mmbert-impresso-mediasources-ner'
REVISION = 'v2.0.0'
DEVICE = None  # Auto-select CUDA when available, then MPS, then CPU. Use -1 to force CPU.
BATCH_SIZE = 32
LOCAL_FILES_ONLY = False
MIN_SCORE = None
FILTER_ANACHRONISTIC = False
RUN_COOKBOOK_PROCESSOR = True

print('Runtime fingerprint')
print(f'impresso_pipelines: {getattr(impresso_pipelines, "__version__", "unknown") if impresso_pipelines else "not imported"}')
print(f'impresso_pipelines file: {getattr(impresso_pipelines, "__file__", "not imported") if impresso_pipelines else "not imported"}')
print(f'MediaSourcesPipeline file: {inspect.getfile(MediaSourcesPipeline)}')
print(f'transformers: {getattr(transformers, "__version__", "not imported") if transformers else "not imported"}')
print(f'torch: {getattr(torch, "__version__", "not imported") if torch else "not imported"}')

pipe = MediaSourcesPipeline(
    model=MODEL_ID,
    revision=REVISION,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    min_score=MIN_SCORE,
    local_files_only=LOCAL_FILES_ONLY,
)

print('Pipeline protocol')
print(f'decoder: {getattr(pipe, "decoder", "unknown")}')
print(f'max_sequence_len: {getattr(pipe, "max_sequence_len", "unknown")}')
print(f'max_annotation_tokens: {getattr(pipe, "max_annotation_tokens", "unknown")}')
print(f'stride: {getattr(pipe, "stride", "unknown")}')
print(f'device: {getattr(pipe, "device", "unknown")}')


## Paste A Rebuilt Content Item

Replace `CONTENT_ITEM_JSON` with one rebuilt content-item JSON object. The notebook reads text from `ft` and uses `ci_id`, `id`, or `c_id` as the display identifier.


In [ ]:
CONTENT_ITEM_JSON = r'''
{"id": "LAB-1859-03-18-a-i0008", "ts": "2026-01-16T11:02:32Z", "pp": [4], "d": "1859-03-18", "cc": true, "olr": true, "st": "newspaper", "sm": "print", "lg": "de", "tp": "ar", "ro": 8, "consolidated": true, "consolidated_reocr_applied": false, "consolidated_ocrqa": 0.83, "lg_original": null, "title": "262", "ppreb": [{"id": "LAB-1859-03-18-a-p0004", "n": 4, "t": [{"c": [1209, 27, 81, 67], "s": 0, "l": 3}, {"c": [180, 114, 53, 57], "s": 4, "l": 3}, {"c": [235, 117, 18, 56], "s": 7, "l": 1}, {"c": [272, 117, 83, 54], "s": 9, "l": 6}, {"c": [384, 123, 50, 56], "s": 16, "l": 4}, {"c": [461, 119, 61, 56], "s": 21, "l": 4}, {"c": [551, 114, 40, 57], "s": 26, "l": 3}, {"c": [611, 117, 77, 56], "s": 30, "l": 4}, {"c": [713, 117, 46, 54], "s": 35, "l": 3}, {"c": [786, 117, 119, 54], "s": 39, "l": 6}, {"c": [934, 119, 137, 54], "s": 46, "l": 10}, {"c": [1100, 117, 119, 54], "s": 57, "l": 8}, {"c": [180, 162, 57, 54], "s": 66, "l": 3}, {"c": [255, 162, 157, 54], "s": 70, "l": 9}, {"c": [432, 166, 63, 55], "s": 80, "l": 4}, {"c": [518, 162, 35, 54], "s": 85, "l": 2}, {"c": [576, 162, 23, 54], "s": 88, "l": 2}, {"c": [599, 162, 16, 54], "s": 91, "l": 1}, {"c": [638, 162, 171, 54], "s": 93, "l": 12}, {"c": [838, 166, 111, 57], "s": 106, "l": 7}, {"c": [949, 166, 8, 57], "s": 113, "l": 1}, {"c": [982, 162, 27, 56], "s": 115, "l": 2}, {"c": [1032, 166, 110, 57], "s": 118, "l": 9}, {"c": [1169, 162, 48, 56], "s": 128, "l": 4}, {"c": [180, 210, 53, 56], "s": 133, "l": 3}, {"c": [251, 210, 36, 56], "s": 137, "l": 2}, {"c": [312, 210, 87, 56], "s": 140, "l": 6}, {"c": [424, 210, 104, 56], "s": 147, "l": 5}, {"c": [555, 210, 44, 56], "s": 153, "l": 3}, {"c": [628, 210, 52, 56], "s": 157, "l": 4}, {"c": [703, 212, 110, 56], "s": 162, "l": 7}, {"c": [844, 210, 52, 56], "s": 170, "l": 3}, {"c": [921, 210, 142, 56], "s": 174, "l": 9}, {"c": [1088, 210, 117, 56], "s": 184, "l": 8}, {"c": [1213, 212, 14, 54], "s": 192, "l": 1}, {"c": [180, 260, 84, 56], "s": 194, "l": 6}, {"c": [285, 258, 68, 56], "s": 201, "l": 4}, {"c": [372, 258, 50, 56], "s": 206, "l": 3}, {"c": [445, 258, 37, 56], "s": 210, "l": 3}, {"c": [505, 258, 200, 56], "s": 214, "l": 12}, {"c": [732, 258, 33, 56], "s": 227, "l": 2}, {"c": [790, 258, 75, 56], "s": 230, "l": 5}, {"c": [884, 258, 123, 56], "s": 236, "l": 8}, {"c": [1032, 262, 60, 56], "s": 245, "l": 4}, {"c": [1115, 262, 25, 54], "s": 250, "l": 2}, {"c": [1165, 258, 56, 56], "s": 253, "l": 4}, {"c": [180, 306, 113, 56], "s": 258, "l": 7}, {"c": [318, 306, 83, 56], "s": 266, "l": 6}, {"c": [405, 306, 17, 56], "s": 272, "l": 1}, {"c": [439, 304, 56, 56], "s": 274, "l": 3}, {"c": [514, 304, 24, 56], "s": 278, "l": 2}, {"c": [561, 308, 73, 56], "s": 281, "l": 5}, {"c": [655, 306, 62, 56], "s": 287, "l": 4}, {"c": [742, 310, 25, 56], "s": 292, "l": 2}, {"c": [792, 306, 44, 56], "s": 295, "l": 3}, {"c": [865, 306, 50, 56], "s": 299, "l": 3}, {"c": [917, 308, 19, 56], "s": 302, "l": 1}, {"c": [955, 308, 50, 56], "s": 304, "l": 3}, {"c": [1034, 312, 37, 56], "s": 308, "l": 3}, {"c": [1098, 308, 48, 56], "s": 312, "l": 3}, {"c": [1175, 306, 42, 56], "s": 316, "l": 3}, {"c": [180, 354, 100, 56], "s": 320, "l": 7}, {"c": [297, 354, 108, 56], "s": 328, "l": 6}, {"c": [428, 354, 100, 56], "s": 335, "l": 7}, {"c": [532, 354, 17, 54], "s": 342, "l": 1}, {"c": [563, 354, 25, 54], "s": 344, "l": 2}, {"c": [613, 354, 44, 54], "s": 347, "l": 3}, {"c": [684, 358, 36, 56], "s": 351, "l": 3}, {"c": [742, 356, 46, 56], "s": 355, "l": 3}, {"c": [809, 358, 87, 56], "s": 359, "l": 5}, {"c": [919, 356, 94, 56], "s": 365, "l": 6}, {"c": [1032, 360, 33, 56], "s": 372, "l": 2}, {"c": [1086, 358, 73, 56], "s": 375, "l": 5}, {"c": [1184, 354, 27, 56], "s": 381, "l": 1, "hy1": true}, {"c": [180, 402, 88, 54], "s": 381, "l": 8, "hy2": true}, {"c": [295, 402, 67, 56], "s": 390, "l": 5}, {"c": [362, 402, 8, 56], "s": 395, "l": 1}, {"c": [249, 449, 17, 55], "s": 397, "l": 1}, {"c": [268, 447, 39, 57], "s": 399, "l": 2}, {"c": [326, 447, 42, 57], "s": 402, "l": 3}, {"c": [391, 449, 131, 57], "s": 406, "l": 7}, {"c": [522, 449, 8, 57], "s": 413, "l": 1}, {"c": [572, 449, 43, 57], "s": 415, "l": 4}, {"c": [643, 454, 33, 56], "s": 420, "l": 3}, {"c": [701, 452, 150, 54], "s": 424, "l": 10}, {"c": [851, 452, 8, 54], "s": 434, "l": 1}, {"c": [882, 452, 17, 54], "s": 436, "l": 1}, {"c": [901, 454, 29, 56], "s": 438, "l": 3}, {"c": [951, 452, 68, 54], "s": 442, "l": 4}, {"c": [1040, 452, 25, 54], "s": 447, "l": 1}, {"c": [1063, 452, 25, 54], "s": 449, "l": 1}, {"c": [1105, 452, 52, 54], "s": 451, "l": 4}, {"c": [1171, 452, 40, 54], "s": 456, "l": 1, "hy1": true}, {"c": [180, 495, 67, 56], "s": 456, "l": 6, "hy2": true}, {"c": [276, 499, 17, 55], "s": 463, "l": 1}, {"c": [293, 499, 46, 55], "s": 465, "l": 4}, {"c": [362, 497, 129, 54], "s": 470, "l": 9}, {"c": [495, 495, 12, 56], "s": 479, "l": 1}, {"c": [526, 497, 46, 54], "s": 481, "l": 3}, {"c": [597, 502, 33, 54], "s": 485, "l": 3}, {"c": [653, 499, 25, 57], "s": 489, "l": 2}, {"c": [678, 499, 21, 57], "s": 492, "l": 1}, {"c": [722, 502, 12, 54], "s": 494, "l": 1}, {"c": [734, 502, 44, 54], "s": 496, "l": 4}, {"c": [797, 499, 156, 57], "s": 501, "l": 10}, {"c": [953, 499, 27, 57], "s": 511, "l": 1}, {"c": [1344, 114, 15, 57], "s": 513, "l": 1}, {"c": [1361, 112, 62, 57], "s": 515, "l": 3}, {"c": [1452, 112, 67, 57], "s": 519, "l": 4}, {"c": [1546, 112, 75, 57], "s": 524, "l": 5}, {"c": [1644, 112, 56, 57], "s": 530, "l": 3}, {"c": [1725, 114, 73, 55], "s": 534, "l": 4}, {"c": [1798, 114, 16, 55], "s": 538, "l": 1}, {"c": [1864, 114, 81, 55], "s": 540, "l": 6}, {"c": [1977, 114, 95, 57], "s": 547, "l": 7}, {"c": [2104, 114, 79, 55], "s": 555, "l": 5}, {"c": [2214, 123, 106, 54], "s": 561, "l": 7}, {"c": [1275, 160, 161, 56], "s": 569, "l": 11}, {"c": [1456, 160, 57, 56], "s": 581, "l": 3}, {"c": [1533, 160, 98, 56], "s": 585, "l": 6}, {"c": [1656, 166, 48, 57], "s": 592, "l": 4}, {"c": [1725, 162, 58, 56], "s": 597, "l": 4}, {"c": [1806, 160, 48, 56], "s": 602, "l": 3}, {"c": [1879, 160, 48, 56], "s": 606, "l": 2}, {"c": [1925, 160, 10, 56], "s": 608, "l": 1}, {"c": [1344, 208, 15, 56], "s": 610, "l": 1}, {"c": [1361, 206, 62, 56], "s": 612, "l": 3}, {"c": [1446, 206, 56, 56], "s": 616, "l": 3}, {"c": [1525, 206, 69, 56], "s": 620, "l": 5}, {"c": [1617, 208, 106, 56], "s": 626, "l": 6}, {"c": [1750, 208, 39, 56], "s": 633, "l": 1}, {"c": [1812, 206, 52, 56], "s": 635, "l": 3}, {"c": [1862, 206, 9, 56], "s": 638, "l": 1}, {"c": [1891, 210, 113, 56], "s": 640, "l": 7}, {"c": [2027, 208, 35, 56], "s": 648, "l": 2}, {"c": [2062, 208, 17, 56], "s": 651, "l": 1}, {"c": [2102, 208, 41, 56], "s": 653, "l": 3}, {"c": [2162, 212, 77, 56], "s": 657, "l": 5}, {"c": [2264, 210, 50, 56], "s": 663, "l": 3}, {"c": [1275, 256, 146, 56], "s": 667, "l": 9}, {"c": [1421, 256, 10, 56], "s": 676, "l": 1}, {"c": [1448, 260, 29, 56], "s": 678, "l": 3}, {"c": [1500, 256, 29, 56], "s": 682, "l": 2}, {"c": [1552, 258, 67, 56], "s": 685, "l": 5}, {"c": [1642, 254, 39, 56], "s": 691, "l": 3}, {"c": [1706, 254, 90, 56], "s": 695, "l": 4}, {"c": [1798, 256, 16, 56], "s": 699, "l": 1}, {"c": [1827, 256, 33, 56], "s": 701, "l": 2}, {"c": [1860, 256, 17, 56], "s": 704, "l": 1}, {"c": [1900, 256, 45, 56], "s": 706, "l": 3}, {"c": [1970, 256, 57, 56], "s": 710, "l": 4}, {"c": [2052, 256, 41, 56], "s": 715, "l": 3}, {"c": [2122, 256, 54, 56], "s": 719, "l": 3}, {"c": [2199, 260, 82, 56], "s": 723, "l": 5}, {"c": [2281, 260, 29, 56], "s": 728, "l": 1}, {"c": [1344, 304, 15, 56], "s": 730, "l": 1}, {"c": [1361, 304, 60, 54], "s": 732, "l": 3}, {"c": [1442, 304, 58, 54], "s": 736, "l": 4}, {"c": [1500, 304, 17, 54], "s": 740, "l": 1}, {"c": [1544, 304, 33, 54], "s": 742, "l": 1}, {"c": [1598, 304, 60, 54], "s": 744, "l": 4}, {"c": [1660, 304, 17, 56], "s": 748, "l": 1}, {"c": [1714, 306, 111, 56], "s": 750, "l": 9}, {"c": [1846, 308, 35, 56], "s": 760, "l": 3}, {"c": [1902, 304, 54, 56], "s": 764, "l": 3}, {"c": [1977, 304, 45, 56], "s": 768, "l": 3}, {"c": [2047, 308, 86, 56], "s": 772, "l": 6}, {"c": [2147, 306, 169, 56], "s": 779, "l": 10}, {"c": [1275, 354, 69, 54], "s": 790, "l": 5}, {"c": [1369, 352, 85, 54], "s": 796, "l": 5}, {"c": [1481, 352, 84, 54], "s": 802, "l": 6}, {"c": [1587, 358, 50, 56], "s": 809, "l": 4}, {"c": [1656, 354, 110, 56], "s": 814, "l": 9}, {"c": [1789, 352, 15, 56], "s": 824, "l": 1}, {"c": [1804, 352, 33, 56], "s": 826, "l": 3}, {"c": [1864, 352, 42, 56], "s": 830, "l": 3}, {"c": [1929, 354, 114, 54], "s": 834, "l": 6}, {"c": [2075, 352, 58, 56], "s": 841, "l": 4}, {"c": [2166, 352, 52, 56], "s": 846, "l": 3}, {"c": [2222, 354, 17, 56], "s": 849, "l": 1}, {"c": [2262, 354, 54, 56], "s": 851, "l": 3}, {"c": [1275, 400, 42, 54], "s": 855, "l": 3}, {"c": [1334, 400, 108, 56], "s": 859, "l": 6}, {"c": [1469, 400, 62, 56], "s": 866, "l": 5}, {"c": [1550, 400, 79, 54], "s": 872, "l": 5}, {"c": [1629, 400, 19, 54], "s": 878, "l": 1}, {"c": [1671, 402, 112, 56], "s": 880, "l": 7}, {"c": [1806, 400, 48, 54], "s": 888, "l": 3}, {"c": [1875, 400, 39, 54], "s": 892, "l": 3}, {"c": [1937, 402, 90, 54], "s": 896, "l": 6}, {"c": [2050, 402, 85, 54], "s": 903, "l": 5}, {"c": [2135, 402, 17, 54], "s": 909, "l": 1}, {"c": [2179, 404, 60, 56], "s": 911, "l": 4}, {"c": [2260, 404, 56, 54], "s": 916, "l": 3}, {"c": [1275, 449, 194, 57], "s": 920, "l": 12}, {"c": [1490, 447, 41, 55], "s": 933, "l": 3}, {"c": [1554, 447, 106, 55], "s": 937, "l": 5}, {"c": [1660, 447, 11, 55], "s": 942, "l": 1}, {"c": [1698, 447, 14, 55], "s": 944, "l": 1}, {"c": [1717, 445, 14, 57], "s": 946, "l": 1}, {"c": [1752, 445, 66, 57], "s": 948, "l": 4}, {"c": [1818, 445, 11, 57], "s": 952, "l": 1}, {"c": [1858, 447, 79, 55], "s": 954, "l": 5}, {"c": [1970, 452, 32, 54], "s": 960, "l": 3}, {"c": [2029, 452, 73, 56], "s": 964, "l": 5}, {"c": [2129, 449, 79, 55], "s": 970, "l": 5}, {"c": [2229, 449, 52, 55], "s": 976, "l": 4}, {"c": [2285, 452, 18, 54], "s": 980, "l": 1}, {"c": [1275, 499, 67, 55], "s": 982, "l": 6}, {"c": [1363, 497, 35, 57], "s": 989, "l": 3}, {"c": [1419, 493, 48, 56], "s": 993, "l": 3}, {"c": [1490, 493, 68, 56], "s": 997, "l": 5}, {"c": [1581, 495, 148, 54], "s": 1003, "l": 8}, {"c": [1752, 495, 23, 54], "s": 1012, "l": 1}, {"c": [1775, 495, 16, 54], "s": 1014, "l": 1}, {"c": [1794, 495, 56, 54], "s": 1016, "l": 4}, {"c": [1850, 495, 16, 54], "s": 1021, "l": 1}, {"c": [1866, 495, 71, 54], "s": 1023, "l": 4}, {"c": [1960, 497, 75, 57], "s": 1028, "l": 4}, {"c": [2037, 495, 13, 56], "s": 1032, "l": 1}, {"c": [2068, 497, 11, 54], "s": 1034, "l": 1}, {"c": [2079, 497, 79, 54], "s": 1035, "l": 5}, {"c": [2158, 497, 10, 54], "s": 1040, "l": 1}, {"c": [2183, 497, 68, 54], "s": 1042, "l": 5}, {"c": [2251, 497, 11, 54], "s": 1047, "l": 1}, {"c": [2262, 497, 10, 54], "s": 1048, "l": 1}], "r": [[1208, 38, 77, 39], [179, 124, 1042, 422], [1275, 121, 1040, 425]]}], "lb": [3, 65, 132, 193, 257, 319, 381, 396, 456, 512, 568, 609, 666, 729, 789, 854, 919, 981, 1049], "pb": [4, 513], "rb": [4, 513], "ft": "262 war, ließen sich auch die Züge des Mannes deutlicher erkennen und gemahnten mich an ei » allbckaxntes Geficht; ja plötzlich trat mir in diesem Manne der alte Schäfer von Reneville entgegen, dessen Bild mir die Erinnernugen an meine Kindheit noch so treu bewahrt hatten, und er stand ganz so vor mir, wie ich ihn vor dreißig Jahren gesehen, ja wic ich ihn nicht wiedei zu sehen erwartet hatte. „ Ei der Tausend! rief ich überrascht; „ ist denn d « alte Nenouz » icht gestorben, daß ich ih » » ster wiedersehe? „ Von wein reden Sie denn? fragte lebhaft meine hübsche Begleiterin und wandte sich nach mir um. „ Von dem alten Nenouz — den, Schäfer vo » der Küste von Reneville; ist es nicht der Mann, de » wir dort vor uns sehen? „ Der dort? O nein, versetzte sie und das schöne Dunkelblau ihres Auges trübte sich plötzlich » vie der Himmel über uns, und der Purpur ihrer Lippe » schwand wie die rothen Wolke » nach dem Verschwiiden der Sonne; „ o nein; Iener ist schon lange todt, setzte sic mit einer mühsamen A » stre » gung hmzu. (Forts, folgt.) "}
'''.strip()

content_item = json.loads(CONTENT_ITEM_JSON)
text = content_item.get('ft') or ''
content_id = content_item.get('ci_id') or content_item.get('id') or content_item.get('c_id') or '<missing id>'
publication_date = content_item.get('date') or content_item.get('d') or content_item.get('year') or content_item.get('publication_date')

if not isinstance(text, str) or not text.strip():
    raise ValueError('The content item must contain non-empty full text in the ft property.')

print(f'id: {content_id}')
print(f'publication_date: {publication_date}')
print(f'characters: {len(text)}')
print(f'text_sha256: {text_sha256(text)}')
print(f'last_120_chars: {text[-120:]}')


## Run Diagnostics


In [ ]:
call_kwargs: dict[str, Any] = {'diagnostics': True}
signature = inspect.signature(pipe.__call__)
if 'publication_date' in signature.parameters:
    call_kwargs['publication_date'] = publication_date
if 'filter_anachronistic' in signature.parameters:
    call_kwargs['filter_anachronistic'] = FILTER_ANACHRONISTIC

result = pipe(text, **call_kwargs)
result


## Display Helpers


In [ ]:
def html_table(rows: list[dict[str, Any]], columns: list[str]) -> HTML:
    if not rows:
        return HTML('<p><em>No rows.</em></p>')
    header = ''.join(f'<th>{html.escape(column)}</th>' for column in columns)
    body_rows = []
    for row in rows:
        cells = []
        for column in columns:
            value = row.get(column, '')
            if isinstance(value, float):
                value = f'{value:.6f}'
            cells.append(f'<td>{html.escape(str(value))}</td>')
        body_rows.append('<tr>' + ''.join(cells) + '</tr>')
    style = '''
    <style>
      table.mediasources { border-collapse: collapse; font-size: 13px; }
      table.mediasources th, table.mediasources td { border: 1px solid #ddd; padding: 4px 7px; vertical-align: top; }
      table.mediasources th { background: #f3f4f6; text-align: left; }
      table.mediasources tr:nth-child(even) { background: #fafafa; }
      .ms-entity { background: #fff3b0; border-bottom: 2px solid #d97706; padding: 0 2px; }
      .ms-text { line-height: 1.8; white-space: pre-wrap; }
    </style>
    '''
    return HTML(style + '<table class="mediasources"><thead><tr>' + header + '</tr></thead><tbody>' + ''.join(body_rows) + '</tbody></table>')

def token_rows(result: dict[str, Any]) -> list[dict[str, Any]]:
    tokens = result.get('tokens', [])
    starts = result.get('token_start_offsets', [])
    stops = result.get('token_end_offsets', [])
    labels = result.get('token_labels', [])
    scores = result.get('token_scores', [])
    return [
        {'i': i, 'token': token, 'start': starts[i], 'stop': stops[i], 'label': labels[i], 'score': scores[i]}
        for i, token in enumerate(tokens)
    ]

def highlighted_text(text: str, entities: list[dict[str, Any]]) -> HTML:
    pieces = []
    cursor = 0
    for entity in sorted(entities, key=lambda item: (item.get('start', 0), item.get('stop', 0))):
        start = int(entity.get('start', cursor))
        stop = int(entity.get('stop', start))
        if start < cursor or stop < start:
            continue
        pieces.append(html.escape(text[cursor:start]))
        label = html.escape(str(entity.get('label', '')))
        qid = html.escape(str(entity.get('wkdata_qid') or ''))
        title = f'{label} {qid}'.strip()
        pieces.append(f'<span class="ms-entity" title="{title}">{html.escape(text[start:stop])}</span>')
        cursor = stop
    pieces.append(html.escape(text[cursor:]))
    return HTML('<div class="ms-text">' + ''.join(pieces) + '</div>')


## Entities


In [ ]:
entities = result.get('entities', [])
display(html_table(entities, ['surface', 'label', 'wkdata_qid', 'start', 'stop', 'score']))
display(highlighted_text(text, entities))


## Summary


In [ ]:
display(html_table(result.get('summary', []), ['uid', 'wkdata_qid', 'score']))


## Token Diagnostics


In [ ]:
display(html_table(token_rows(result), ['i', 'token', 'start', 'stop', 'label', 'score']))


## Cookbook Processor Output

This runs the same `MediaSourcesProcessor` path used by the cookbook CLI for one pasted content item. Use this when you need to compare raw pipeline diagnostics with the exact cookbook `nes` output shape.


In [ ]:
cookbook_row = None

if RUN_COOKBOOK_PROCESSOR:
    with tempfile.TemporaryDirectory(prefix="mediasources-notebook-") as tmpdir:
        tmpdir_path = Path(tmpdir)
        input_path = tmpdir_path / "input.jsonl"
        output_path = tmpdir_path / "output.jsonl"
        log_path = tmpdir_path / "run.log"
        input_path.write_text(json.dumps(content_item, ensure_ascii=False) + "\n", encoding="utf-8")

        processor = MediaSourcesProcessor(
            input_file=str(input_path),
            output_file=str(output_path),
            hf_model=MODEL_ID,
            revision=REVISION,
            batch_size=BATCH_SIZE,
            outer_batch_size=1,
            device=DEVICE,
            min_score=MIN_SCORE,
            filter_anachronistic=FILTER_ANACHRONISTIC,
            local_files_only=LOCAL_FILES_ONLY,
            diagnostics=True,
            write_empty=True,
            log_level="DEBUG",
            log_file=str(log_path),
        )
        processor.run()

        rows = [json.loads(line) for line in output_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if len(rows) != 1:
            raise RuntimeError(f"Expected one cookbook output row, got {len(rows)}")
        cookbook_row = rows[0]

        print("Cookbook DEBUG log")
        print(log_path.read_text(encoding="utf-8"))

cookbook_row


## Cookbook Entities


In [ ]:
if cookbook_row is None:
    print("RUN_COOKBOOK_PROCESSOR is False")
else:
    display(html_table(
        cookbook_row.get("nes", []),
        ["surface", "fine_grained_type", "wkdata_qid", "lOffset", "rOffset", "confidence_ner", "start_year"],
    ))


## Cookbook Token Diagnostics


In [ ]:
if cookbook_row is None:
    print("RUN_COOKBOOK_PROCESSOR is False")
else:
    diagnostics = cookbook_row.get("diagnostics", {})
    cookbook_token_result = {
        "tokens": diagnostics.get("tokens", []),
        "token_start_offsets": diagnostics.get("token_start_offsets", []),
        "token_end_offsets": diagnostics.get("token_end_offsets", []),
        "token_labels": diagnostics.get("token_labels", []),
        "token_scores": diagnostics.get("token_scores", []),
    }
    display(html_table(token_rows(cookbook_token_result), ["i", "token", "start", "stop", "label", "score"]))


## Cookbook JSON Result


In [ ]:
if cookbook_row is None:
    print("RUN_COOKBOOK_PROCESSOR is False")
else:
    print(json.dumps(cookbook_row, ensure_ascii=False, indent=2))


## JSON Result


In [ ]:
print(json.dumps(result, ensure_ascii=False, indent=2))
